# Анализ результатов A/B-тестирования

## 1. Цели исследования.



Проверить, приведёт ли упрощение интерфейса интернет-магазина BitMotion Kit к увеличению конверсии зарегистрированных пользователей в покупателей на три процентных пункта в течение семи дней после регистрации.

Для этого было проведено A/B-тестирование: контрольная группа A использовала старую версию сайта, а тестовая группа B — новую, с упрощённым интерфейсом. Задача анализа — оценить, достигнут ли целевой эффект, и определить статистическую значимость полученных результатов.

## 2. Загрузка данных, оценка их целостности.


In [10]:
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

In [11]:
# Оцениваем визуально имеющиеся данные participants
display(participants.head())
# Смотрим информацию о каждом столбце таблицы participants: тип данных, количество непустых значений 
display(participants.info())

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB


None

In [12]:
# Проверяем пропуски 
display(participants.isnull().sum())

user_id    0
group      0
ab_test    0
device     0
dtype: int64

In [13]:
# Проверяем дубликаты
display(participants.duplicated(subset='user_id').sum())

887

In [14]:
# Проверяем количество пользователей по всем ab_test
display(participants['ab_test'].value_counts())

interface_eu_test          10850
recommender_system_test     3675
Name: ab_test, dtype: int64

In [15]:
# Проверяем количество пользователей по названию ab_test - interface_eu_test
participants_eu = participants[participants['ab_test'] == 'interface_eu_test'].copy()
print(f"Участников в interface_eu_test: {len(participants_eu)}")

Участников в interface_eu_test: 10850


In [16]:
# Оцениваем визуально имеющиеся данные events
display(events.head())
# Смотрим информацию о каждом столбце таблицы events: тип данных, количество непустых значений
display(events.info())

,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB


None

In [17]:
# Проверяем пропуски 
display(events.isnull().sum())

user_id            0
event_dt           0
event_name         0
details       538264
dtype: int64

In [18]:
# Проверяем дубликаты
duplicates = events.duplicated().sum()
print(f'Полных дубликатов строк: {duplicates}')

Полных дубликатов строк: 36318


In [19]:
# Смотрим типы событий
display(events['event_name'].value_counts())

login                                 248285
product_page                          195606
registration                          144183
purchase                              104836
product_cart                           94373
End of Black Friday Ads Campaign           1
Start of CIS New Year Gift Lottery         1
Start of Christmas&New Year Promo          1
Name: event_name, dtype: int64

## 3. По таблице `ab_test_participants` оцениваем корректность проведения теста:

   3\.1 Выделим пользователей, участвующих в тесте, и проверим:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [20]:
# Фильтруем нужный тест
test_name = 'interface_eu_test'
test = participants[participants['ab_test'] == test_name].copy()
print(f"ПРОВЕРКА КОРРЕКТНОСТИ ТЕСТА: {test_name}")

# 1. Соответствие ТЗ
print("\n1. ГРУППЫ ТЕСТА:")
print(f"   Группы: {sorted(test['group'].unique())}")

# 2. Равномерность распределения
print("\n2. РАСПРЕДЕЛЕНИЕ ПО ГРУППАМ:")
counts = test['group'].value_counts()
print(f"   Группа A: {counts['A']} ({counts['A']/len(test)*100:.1f}%)")
print(f"   Группа B: {counts['B']} ({counts['B']/len(test)*100:.1f}%)")

# 3. Пересечения с другими тестами
print("\n3. ПЕРЕСЕЧЕНИЯ С ДРУГИМИ ТЕСТАМИ:")
# Сколько тестов у каждого пользователя
tests_per_user = participants.groupby('user_id')['ab_test'].nunique()
# Пользователи теста, которые участвуют еще где-то
bad_users = set(test['user_id']) & set(tests_per_user[tests_per_user > 1].index)
print(f"   Пользователей в нескольких тестах: {len(bad_users)}")

# 4. ИТОГ
print("\n" + "="*50)
if len(bad_users) > 0:
    print("ВЫВОД: ТЕСТ ПРОВЕДЕН НЕКОРРЕКТНО")
    print(f"→ Нужно исключить {len(bad_users)} пользователей из анализа")
else:
    print("ВЫВОД: ТЕСТ ПРОВЕДЕН КОРРЕКТНО")

ПРОВЕРКА КОРРЕКТНОСТИ ТЕСТА: interface_eu_test

1. ГРУППЫ ТЕСТА:
   Группы: ['A', 'B']

2. РАСПРЕДЕЛЕНИЕ ПО ГРУППАМ:
   Группа A: 5383 (49.6%)
   Группа B: 5467 (50.4%)

3. ПЕРЕСЕЧЕНИЯ С ДРУГИМИ ТЕСТАМИ:
   Пользователей в нескольких тестах: 887

ВЫВОД: ТЕСТ ПРОВЕДЕН НЕКОРРЕКТНО
→ Нужно исключить 887 пользователей из анализа


3\.2 Проанализируем данные о пользовательской активности по таблице `ab_test_events`:

- оставим только события, связанные с участвующими в изучаемом тесте пользователями;

In [21]:
print("ФИЛЬТРАЦИЯ СОБЫТИЙ ДЛЯ ТЕСТА interface_eu_test")
print("="*50)

# 1. Получаем список "чистых" пользователей теста (исключая пересекающихся)
tests_per_user = participants.groupby('user_id')['ab_test'].nunique()
bad_users = tests_per_user[tests_per_user > 1].index

# Чистые пользователи теста
clean_users = participants[
    (participants['ab_test'] == 'interface_eu_test') & 
    (~participants['user_id'].isin(bad_users))
]['user_id'].unique()

print(f"Чистых пользователей в тесте: {len(clean_users)}")

# 2. Оставляем только события этих пользователей
events_clean = events[events['user_id'].isin(clean_users)].copy()

# 3. Убираем мусорные события (GLOBAL)
events_clean = events_clean[~events_clean['user_id'].astype(str).str.contains('GLOBAL', na=False)]

print(f"\nСобытий до фильтрации: {len(events)}")
print(f"Событий после фильтрации: {len(events_clean)}")

# 4. Проверяем типы событий
print(f"\nТипы событий у чистых пользователей:")
print(events_clean['event_name'].value_counts())



ФИЛЬТРАЦИЯ СОБЫТИЙ ДЛЯ ТЕСТА interface_eu_test
Чистых пользователей в тесте: 9963

Событий до фильтрации: 787286
Событий после фильтрации: 73815

Типы событий у чистых пользователей:
login           27360
product_page    17614
registration     9963
purchase         9487
product_cart     9391
Name: event_name, dtype: int64


- определим горизонт анализа: рассчитаем время (лайфтайм) совершения события пользователем после регистрации и оставим только те события, которые были выполнены в течение первых семи дней с момента регистрации;

In [22]:
# 1. Сначала фильтруем события для чистых пользователей теста
events_test = events_clean[events_clean['user_id'].isin(clean_users)].copy()

# 2. Находим дату регистрации каждого пользователя
registration = events_test[events_test['event_name'] == 'registration'][['user_id', 'event_dt']]
registration.columns = ['user_id', 'reg_dt']

# 3. Присоединяем дату регистрации к событиям
events_with_reg = events_test.merge(registration, on='user_id', how='left')

# 4. Рассчитываем день события (лайфтайм)
events_with_reg['day'] = (events_with_reg['event_dt'] - events_with_reg['reg_dt']).dt.days

# 5. Оставляем только события первых 7 дней (0-6 = 7 дней)
events_7days = events_with_reg[events_with_reg['day'] <= 6].copy()

# 6. Проверяем результат
print(f"Событий до фильтрации: {len(events_test)}")
print(f"Событий в первые 7 дней: {len(events_7days)}")
print(f"Удалено событий после 7 дней: {len(events_test) - len(events_7days)}")

# Проверяем распределение по дням
print("\nРаспределение событий по дням (0 = день регистрации):")
print(events_7days['day'].value_counts().sort_index())

Событий до фильтрации: 73815
Событий в первые 7 дней: 63805
Удалено событий после 7 дней: 10010

Распределение событий по дням (0 = день регистрации):
0    37892
1     8047
2     5541
3     4103
4     3240
5     2752
6     2230
Name: day, dtype: int64


Оценим достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [23]:
print("="*60)
print("ОЦЕНКА ДОСТАТОЧНОСТИ ВЫБОРКИ")
print("="*60)

# Параметры
baseline, effect = 0.30, 0.03
alpha, power = 0.05, 0.80
alternative = baseline + effect

# Необходимый размер выборки 
effect_size = proportion_effectsize(baseline, alternative)
n_required = math.ceil(NormalIndPower().solve_power(effect_size, alpha=alpha, power=power, alternative='two-sided'))

print(f"Цель: +{effect*100:.1f} п.п. (30% → {alternative*100:.0f}%)")
print(f"Требуется на группу: {n_required} пользователей")

# Фактические данные
clean_participants = participants[
    (participants['ab_test'] == 'interface_eu_test') & 
    (~participants['user_id'].isin(bad_users))
]
actual_a = len(clean_participants[clean_participants['group'] == 'A'])
actual_b = len(clean_participants[clean_participants['group'] == 'B'])

print(f"Фактически: A = {actual_a}, B = {actual_b}")

# Оценка и минимальный обнаруживаемый эффект
if actual_a >= n_required and actual_b >= n_required:
    print(f"Выборка ДОСТАТОЧНА")
else:
    print(f" Не хватает {n_required - min(actual_a, actual_b)} чел на группу")
    
    # MDE для текущей выборки
    from scipy.optimize import brentq
    def find_mde(n, p1=baseline):
        def func(diff):
            p2 = p1 + diff
            if p2 <= 0 or p2 >= 1: return 1
            effect = proportion_effectsize(p1, p2)
            return NormalIndPower().power(effect, nobs1=n, alpha=alpha, alternative='two-sided') - power
        return brentq(func, 0.001, 0.5-p1)
    
    mde = find_mde(min(actual_a, actual_b))
    print(f"  Текущая выборка может обнаружить эффект > {mde*100:.1f} п.п.")
    print(f"  Целевой эффект: +{effect*100:.1f} п.п. — {'' if mde <= effect else ''}")

print("="*60)

ОЦЕНКА ДОСТАТОЧНОСТИ ВЫБОРКИ
Цель: +3.0 п.п. (30% → 33%)
Требуется на группу: 3762 пользователей
Фактически: A = 4952, B = 5011
Выборка ДОСТАТОЧНА


- рассчитаем для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [24]:
# Получаем чистых пользователей с их группами
clean_participants = participants[
    (participants['ab_test'] == 'interface_eu_test') & 
    (~participants['user_id'].isin(bad_users))
]

# Находим пользователей, совершивших покупку в первые 7 дней
buyers_7d = set(events_7days[events_7days['event_name'] == 'purchase']['user_id'])

for group in ['A', 'B']:
    users = set(clean_participants[clean_participants['group'] == group]['user_id'])
    buyers = len(users & buyers_7d)
    print(f"Группа {group}: {buyers} сделали покупку / {len(users)} всего посетителей = {buyers/len(users)*100:.2f}%")

Группа A: 1377 сделали покупку / 4952 всего посетителей = 27.81%
Группа B: 1480 сделали покупку / 5011 всего посетителей = 29.54%


- сделаем предварительный общий вывод об изменении пользовательской активности в тестовой группе по сравнению с контрольной.

In [25]:
# Определяем переменные
conv_a = 27.81
conv_b = 29.54
diff = conv_b - conv_a  # 1.73
target = 3.0

print("ПРЕДВАРИТЕЛЬНЫЙ ВЫВОД")
print("="*40)
print(f"Конверсия: A={conv_a:.2f}% → B={conv_b:.2f}%")
print(f"Изменение: +{diff:.2f} п.п. (цель: +{target:.1f} п.п.)")
print(f"Эффект: {'ПОЛОЖИТЕЛЬНЫЙ' if diff>0 else 'ОТРИЦАТЕЛЬНЫЙ'}")
print(f"Цель: {'НЕ ДОСТИГНУТА' if diff<target else 'ДОСТИГНУТА'}")
print("\nВывод: Новый интерфейс улучшил конверсию, но ниже целевого порога")

ПРЕДВАРИТЕЛЬНЫЙ ВЫВОД
Конверсия: A=27.81% → B=29.54%
Изменение: +1.73 п.п. (цель: +3.0 п.п.)
Эффект: ПОЛОЖИТЕЛЬНЫЙ
Цель: НЕ ДОСТИГНУТА

Вывод: Новый интерфейс улучшил конверсию, но ниже целевого порога


## 4. Оценка результатов A/B-тестирования:

- Проверим изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

In [26]:
print("="*50)
print("СТАТИСТИЧЕСКАЯ ОЦЕНКА A/B ТЕСТА")
print("="*50)

# Данные
users_a, users_b = 4952, 5011
purchases_a, purchases_b = 1377, 1480

conv_a = purchases_a / users_a  # 27.81%
conv_b = purchases_b / users_b  # 29.54%
diff = (conv_b - conv_a) * 100  # 1.73 п.п.

print(f"\n ДАННЫЕ:")
print(f"  Группа A: {purchases_a}/{users_a} = {conv_a*100:.2f}%")
print(f"  Группа B: {purchases_b}/{users_b} = {conv_b*100:.2f}%")
print(f"  Разница: +{diff:.2f} п.п.")

# Z-тест (односторонний)
z_stat, p_value = proportions_ztest(
    count=[purchases_b, purchases_a],
    nobs=[users_b, users_a],
    alternative='larger'
)

print(f"\n СТАТИСТИЧЕСКИЙ ТЕСТ:")
print(f"  H₀: конверсия B ≤ конверсии A")
print(f"  H₁: конверсия B > конверсии A")
print(f"  p-value: {p_value:.4f}")

# Вывод
alpha = 0.05
print(f"\n РЕЗУЛЬТАТ:")

if p_value < alpha:
    print(f"  p-value ({p_value:.4f}) < α ({alpha})")
    print(f"  → Статистически значимое улучшение ЕСТЬ")
else:
    print(f"  p-value ({p_value:.4f}) ≥ α ({alpha})")
    print(f"  → Статистически значимого улучшения НЕТ")

# Проверка цели по ТЗ
target = 3.0
print(f"\n ЦЕЛЬ ПО ТЗ (+{target} п.п.):")
if diff >= target:
    print(f" Достигнута ({diff:.2f} >= {target})")
else:
    print(f" НЕ достигнута ({diff:.2f} < {target})")


СТАТИСТИЧЕСКАЯ ОЦЕНКА A/B ТЕСТА

 ДАННЫЕ:
  Группа A: 1377/4952 = 27.81%
  Группа B: 1480/5011 = 29.54%
  Разница: +1.73 п.п.

 СТАТИСТИЧЕСКИЙ ТЕСТ:
  H₀: конверсия B ≤ конверсии A
  H₁: конверсия B > конверсии A
  p-value: 0.0283

 РЕЗУЛЬТАТ:
  p-value (0.0283) < α (0.05)
  → Статистически значимое улучшение ЕСТЬ

 ЦЕЛЬ ПО ТЗ (+3.0 п.п.):
 НЕ достигнута (1.73 < 3.0)


Ожидаемый эффект в изменении конверсии достигнут не был. По техническому заданию требовалось увеличение конверсии на 3 процентных пункта, однако по итогам теста фактический прирост составил всего 1.73 процентных пункта.

Тем не менее, новый интерфейс показал статистически значимое улучшение. Значение p-value составило 0.0283, что меньше уровня значимости 0.05. Это означает, что наблюдаемое различие между группами не является случайным, и новый интерфейс действительно работает лучше контрольной версии.

Конверсия в тестовой группе B составила 29.54%, что выше контрольной группы A с показателем 27.81%. Таким образом, упрощение интерфейса привело к положительному эффекту, но его величина оказалась почти в два раза ниже запланированной.

При проведении теста были выявлены нарушения. 887 пользователей (8.2% от всех участников) одновременно участвовали в двух A/B-тестах — interface_eu_test и recommender_system_test. Это нарушает принцип чистоты эксперимента, поскольку поведение таких пользователей могло быть подвержено влиянию сразу двух тестов. Данные пользователи были исключены из анализа.

Выборка оказалась достаточной для проведения теста. В каждой группе после очистки осталось около 5000 пользователей, что превышает требуемый минимальный размер в 3762 человека на группу для обнаружения эффекта в 3 процентных пункта.

`Итоговая рекомендация: новый интерфейс можно внедрять, но с осторожностью. Он действительно повышает конверсию, однако эффект оказался ниже запланированного. Бизнесу стоит либо пересмотреть ожидания по эффекту в сторону уменьшения, либо доработать интерфейс и провести повторное A/B-тестирование для достижения целевого показателя в три процентных пункта.`